In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "paths.py").is_file():
    raise FileNotFoundError("Could not locate the project root")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from paths import *
from heatwave import limit_to_study_period
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "paths.py").is_file():
    raise FileNotFoundError("Could not locate the project root")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from paths import *
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "paths.py").is_file():
    raise FileNotFoundError("Could not locate the project root")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from paths import *






In [ ]:
# System
import os
import glob
import sys
from typing import Optional

# Data
import pandas as pd
import numpy as np
import xarray as xr

# Visualization
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns





In [ ]:
base_path = str(ERA5_CAPITALS_DIR)
cities = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]





In [ ]:
# Functions
def compute_noaa_hwi(T_celsius, RH):
    """Compute NOAA Heatwave Index (Heat Index) given T in Celsius and RH (%)"""
    T = T_celsius * 9/5 + 32  # Converte para Fahrenheit
    HI = (-42.379 + 2.04901523*T + 10.14333127*RH - 0.22475541*T*RH
          - 0.00683783*T**2 - 0.05481717*RH**2 + 0.00122874*T**2*RH
          + 0.00085282*T*RH**2 - 0.00000199*T**2*RH**2)
    return HI


def compute_omm_hwi(df,
                    var_t2m='t2m',

                    var_anomaly='anomaly',
                    var_mask='mask',
                    omm_hwi='omm_hwi'):

    df = df.reset_index()
    df.set_index('time', inplace=True)
    df.index = pd.to_datetime(df.index)

    # Monthly Climatology (1961–1990)
    ref = df.loc['1961-01-01':'1990-12-31', var_t2m]
    clim_ref = ref.groupby(ref.index.month).mean()

    # Anomaly
    df[var_anomaly] = df[var_t2m] - df.index.month.map(clim_ref)
    df[var_mask] = df[var_anomaly].where(df[var_anomaly] > 5, other=0)
    df[omm_hwi] = df[var_mask] - 5
    df[omm_hwi] = df[omm_hwi].where(df[omm_hwi] > 0, other=0)

    return df


def count_episodes_per_month(series: pd.Series, min_length: int = 5, name: Optional[str] = None) -> pd.Series:
    """
    Conta, por mês, episódios NÃO-SOBREPOSTOS de >= min_length dias consecutivos com valor > 0.
    - Reseta na virada do mês e em qualquer dia <= 0 (ou NaN).
    - Retorna série mensal indexada no 1º dia do mês (int64).
    """
    if series is None or series.size == 0:
        return pd.Series(dtype="int64", name=(name or "episodes"))

    s = series.copy()
    s.index = pd.to_datetime(s.index)
    s = s.sort_index()
    s = s.gt(0).fillna(False)  # True = dia válido; False = quebra

    if s.empty:
        return pd.Series(dtype="int64", name=(name or f"{series.name}_ep{min_length}" if series.name else f"ep{min_length}"))

    # Série mensal alinhada ao índice diário (agora sim dá pra usar .ne/.shift)
    month_s = pd.Series(s.index.to_period("M"), index=s.index)
    month_start = month_s.ne(month_s.shift()).fillna(True)

    # Quebra grupos em: inícios de mês OU dias False
    grp = ((~s) | month_start).cumsum()

    # Comprimento da run True dentro de cada grupo
    run_len_running = s.groupby(grp).cumsum()        # 1,2,3... nas runs True; 0 nos False
    run_len_per_grp = run_len_running.groupby(grp).max().fillna(0).astype("int64")

    # Episódios por grupo (não-sobrepostos)
    episodes_per_grp = (run_len_per_grp // min_length)

    # Mês (1º dia) de cada grupo
    grp_month = month_s.groupby(grp).first().dt.to_timestamp(how="start")

    # Soma por mês
    monthly = episodes_per_grp.groupby(grp_month).sum()

    # Índice mensal contínuo: do 1º mês ao último presente no índice da série
    start_ms = s.index.min().to_period("M").to_timestamp(how="start")
    end_ms   = s.index.max().to_period("M").to_timestamp(how="start")
    full_idx = pd.date_range(start_ms, end_ms, freq="MS")

    out = monthly.reindex(full_idx, fill_value=0).astype("int64")
    out.name = name or (f"{series.name}_ep{min_length}" if series.name else f"ep{min_length}")
    return out


def diary_ind_prod(path):
    df = pd.read_parquet(path)
    df = df.reset_index()
    df.set_index("time", inplace=True)
    df.index = pd.to_datetime(df.index)

    # Grouping the data by day and calculating the sum of the index values
    diary_sum = df.resample('D')['XHWI'].sum().rename('diary_sum')
    # Calculating the total number of hours the index is non-zero for each day
    non_zero_hours = df['XHWI'].apply(lambda x: 1 if x != 0 else 0).resample('D').sum().rename('non_zero_hours')
    # Ind Prod XHWI
    ind_prod = (diary_sum * non_zero_hours).rename('Ind_Prod_Diary')

    # Temperature
    t2m_mean = df.resample("D")["t2m"].mean().rename('t2m_mean')
    t2m_max = df.resample("D")["t2m"].max().rename('t2m_max')
    t2m_min = df.resample("D")["t2m"].min().rename('t2m_min')

    # Relative Humidity
    r = df.resample("D")["r"].mean()

    # NOAA HWI
    noaa_daily_mean = df.resample('D')["NOAA_HWI"].mean().rename('NOAA_HWI_mean')
    noaa_daily_max = df.resample('D')["NOAA_HWI"].max().rename('NOAA_HWI_max')
    noaa_daily_min = df.resample('D')["NOAA_HWI"].min().rename('NOAA_HWI_min')

    df["caution"]          = ((df["NOAA_HWI"] >= 80) & (df["NOAA_HWI"] < 90)).astype(int)
    df["extreme_caution"]  = ((df["NOAA_HWI"] >= 90) & (df["NOAA_HWI"] < 103)).astype(int)
    df["danger"]           = ((df["NOAA_HWI"] >= 103) & (df["NOAA_HWI"] < 125)).astype(int)
    df["extreme_danger"]   = (df["NOAA_HWI"] >= 125).astype(int)

    df_hwi_daily_counts = df[[
                              "caution",
                              "extreme_caution",
                              "danger",
                              "extreme_danger"
                              ]].resample("D").sum()

    # Concat Variables
    df_I = pd.concat(
                     [
                      t2m_mean, t2m_max, t2m_min,  # temperature
                      r,  # Relative Humidity
                      diary_sum, non_zero_hours, ind_prod,  # XHWI
                      noaa_daily_mean, noaa_daily_max, noaa_daily_min,  # NOAA HWI
                      df_hwi_daily_counts
                      ],
                      axis=1
                     )

    df_daily = df_I.copy()
    #display(df_daily)
    # OMM HWI
    df_daily = compute_omm_hwi(df_daily,
                               var_t2m='t2m_max',

                               var_anomaly='anomaly_max',
                               var_mask='mask_max',
                               omm_hwi='omm_hwi_max')

    df_daily.index.name = "time"
    df_daily = df_daily.reset_index()
    df_daily = df_daily.sort_values('time')
    return df_daily


def monthly_ind_prod(path):
    df = pd.read_parquet(path)
    df = df.reset_index()
    df.set_index("time", inplace=True)
    df.index = pd.to_datetime(df.index)

    # Ind Prod XHWI
    monthly_sum = df.resample('MS')['Ind_Prod_Diary'].sum().rename('Ind_Prod_Monthly')

    df["ind_prod_monthly_qtd"] = ((df['Ind_Prod_Diary'] > 0).astype(int))
    df_xhwi_monthly_qtd = df['ind_prod_monthly_qtd'].resample("MS").sum().rename("ind_prod_monthly_qtd")
    df_ind_prod_persist3 = count_episodes_per_month(df["ind_prod_monthly_qtd"], 3, name="ind_prod_3episodes")
    df_ind_prod_persist5 = count_episodes_per_month(df["ind_prod_monthly_qtd"], 5, name="ind_prod_5episodes")


    # Temperature
    t2m_mean = df.resample("MS")["t2m_mean"].mean().rename('t2m_mean')
    t2m_max = df.resample("MS")["t2m_max"].max().rename('t2m_max')
    t2m_min = df.resample("MS")["t2m_min"].min().rename('t2m_min')

    # Relative Humidity
    r = df.resample("MS")["r"].mean().rename('rh_monthly_mean')

    # NOAA HWI
    noaa_monthly_mean = df.resample('MS')["NOAA_HWI_mean"].mean().rename('NOAA_HWI_mean')
    noaa_monthly_max = df.resample('MS')["NOAA_HWI_max"].max().rename('NOAA_HWI_max')
    noaa_monthly_min = df.resample('MS')["NOAA_HWI_min"].min().rename('NOAA_HWI_min')

    caution         = df.resample('MS')["caution"].sum().rename('caution_qtd_hr_month')
    extreme_caution = df.resample('MS')["extreme_caution"].sum().rename('extreme_caution_qtd_hr_month')
    danger          = df.resample('MS')["danger"].sum().rename('danger_qtd_hr_month')
    extreme_danger  = df.resample('MS')["extreme_danger"].sum().rename('extreme_danger_qtd_hr_month')


    df["noaa_qtd"] = ((df[["caution", "extreme_caution", "danger", "extreme_danger"]] > 0).any(axis=1).astype(int))
    df_noaa_hwi_monthly_counts = df['noaa_qtd'].resample("MS").sum().rename("noaa_qtd_monthly")
    df_noaa_persist3 = count_episodes_per_month(df["noaa_qtd"], 3, name="noaa_3episodes")
    df_noaa_persist5 = count_episodes_per_month(df["noaa_qtd"], 5, name="noaa_5episodes")


    # OMM
    df["omm_qtd"] = ((df['omm_hwi_max'] > 0).astype(int))
    df_omm_hwi_monthly_counts = df['omm_qtd'].resample("MS").sum().rename("omm_qtd_monthly")
    df_omm_persist3 = count_episodes_per_month(df["omm_qtd"], 3, name="omm_3episodes")
    df_omm_persist5 = count_episodes_per_month(df["omm_qtd"], 5, name="omm_5episodes")

    # Concat Variables
    df_monthly = pd.concat(
                           [
                            t2m_mean, t2m_max, t2m_min,  # temperature
                            r,  # Relative Humidity
                            monthly_sum, df_xhwi_monthly_qtd,  # XHWI
                            df_ind_prod_persist3, df_ind_prod_persist5,  # XHWI episode persist
                            noaa_monthly_mean, noaa_monthly_max, noaa_monthly_min,  # NOAA HWI
                            caution, extreme_caution, danger, extreme_danger,  # NOAA HWI qtd of hours in month
                            df_noaa_hwi_monthly_counts,  # NOAA HWI qtd conditions
                            df_noaa_persist3, df_noaa_persist5,  # NOAA episode persist
                            df_omm_hwi_monthly_counts,  # OMM HWI qtd conditions
                            df_omm_persist3, df_omm_persist5  # OMM HWI episode persist
                            ],
                            axis=1
                           )

    df_monthly.index.name = "time"
    df_monthly = df_monthly.reset_index()
    df_monthly = df_monthly.sort_values('time')
    return df_monthly





In [ ]:
# Calculating NOAA HeatIndex
for city in cities:
    city_path = os.path.join(base_path, city)
    csv_files = sorted(glob.glob(os.path.join(city_path, f"{city}_heatwave_ERA5_month_*.csv")))

    if not csv_files:
        print(f"⚠️ Nenhum arquivo encontrado para {city}")
        continue

    all_months = []

    for f in csv_files:
        print(f"📂 Lendo: {os.path.basename(f)}")

        df = pd.read_csv(f)
        df["time"] = pd.to_datetime(df['time'])

        # Verifica se as colunas esperadas existem
        if not {'t2m', 'r'}.issubset(df.columns):
            raise ValueError(f"Arquivo {f} não contém colunas 't2m' e 'r'.")

        # NOAA Heatwave Index
        df['NOAA_HWI'] = compute_noaa_hwi(df['t2m'], df['r'])

        # OMM Heateave Index
        df = compute_omm_hwi(df)

        df = df.reset_index()
        all_months.append(df)

    # Concatena todos os meses
    df_all = pd.concat(all_months, ignore_index=True)
    df_all = df_all.sort_values("time")
    df_all = limit_to_study_period(df_all)

    # Salva em Parquet
    output_file = str(ERA5_HOURLY_DIR / city / f"{city}_heatwave_ERA5_hourly.parquet")
    df_all.to_parquet(output_file, index=False)

    print(f"✅ Arquivo salvo: {output_file}")





In [ ]:
# Cálculo da agregação diária

for city in cities:
    city_path = os.path.join(base_path, city)
    pqt_files = sorted((ERA5_HOURLY_DIR / city).glob(f"{city}_heatwave_ERA5_hourly.parquet"))
    if not pqt_files:
        print(f"⚠️ Nenhum arquivo encontrado para {city}")
        continue

    for f in pqt_files:
        print(f"📂 Lendo: {os.path.basename(f)}")

        df_diary = diary_ind_prod(f)
        df_diary = df_diary.reset_index()
        df_diary = df_diary.sort_values('time')
        df_diary = df_diary.drop(columns='index')

        # Salva em Parquet
        output_file = str(ERA5_DAILY_DIR / city / f"{city}_heatwave_ERA5_diary.parquet")
        df_diary.to_parquet(output_file, index=False)

        print(f"✅ Arquivo salvo: {output_file}")





In [ ]:
pth = str(ERA5_DAILY_DIR / "Rio de Janeiro" / "Rio de Janeiro_heatwave_ERA5_diary.parquet")

pd.set_option("display.max_columns", None)
df = pd.read_parquet(pth)
#df.loc[df['Ind_Prod_Diary'] > 5]
display(df)





In [ ]:
# Cálculo da agregação mensal

for city in cities:
    city_path = os.path.join(base_path, city)
    pqt_files = sorted((ERA5_DAILY_DIR / city).glob(f"{city}_heatwave_ERA5_diary.parquet"))
    if not pqt_files:
        print(f"⚠️ Nenhum arquivo encontrado para {city}")
        continue

    for f in pqt_files:
        print(f"📂 Lendo: {os.path.basename(f)}")

        df_monthly = monthly_ind_prod(f)
        df_monthly = df_monthly.reset_index()
        df_monthly = df_monthly.sort_values('time')
        df_monthly = df_monthly.drop(columns='index')

        # Salva em Parquet
        output_file = str(ERA5_MONTHLY_DIR / city / f"{city}_heatwave_ERA5_monthly.parquet")
        df_monthly.to_parquet(output_file, index=False)

        print(f"✅ Arquivo salvo: {output_file}")





In [ ]:
pth = str(ERA5_MONTHLY_DIR / "Rio de Janeiro" / "Rio de Janeiro_heatwave_ERA5_monthly.parquet")

pd.set_option("display.max_columns", None)
df = pd.read_parquet(pth)
#df.loc[df['Ind_Prod_Diary'] > 5]
display(df)





# Novos arquivos de agregações e cidades

In [ ]:
from pathlib import Path
import unicodedata





In [ ]:
# Mantém o dicionário original (chave = HW_*, valor = Cidade)
CODE_TO_CAPITAL = {
                  "HW_SE": "Aracaju",
                  "HW_PA": "Belém",
                  "HW_MG": "Belo Horizonte",
                  "HW_RR": "Boa Vista",
                  "HW_DF": "Brasília",
                  "HW_MS": "Campo Grande",
                  "HW_MT": "Cuiabá",
                  "HW_PR": "Curitiba",
                  "HW_SC": "Florianópolis",
                  "HW_CE": "Fortaleza",
                  "HW_GO": "Goiânia",
                  "HW_PB": "João Pessoa",
                  "HW_AP": "Macapá",
                  "HW_AL": "Maceió",
                  "HW_AM": "Manaus",
                  "HW_RN": "Natal",
                  "HW_TO": "Palmas",
                  "HW_RS": "Porto Alegre",
                  "HW_RO": "Porto Velho",
                  "HW_PE": "Recife",
                  "HW_AC": "Rio Branco",
                  "HW_RJ": "Rio de Janeiro",
                  "HW_BA": "Salvador",
                  "HW_MA": "São Luís",
                  "HW_SP": "São Paulo",
                  "HW_PI": "Teresina",
                  "HW_ES": "Vitória"
                  }

METRICS = [
          "Ind_Prod_Monthly",
          "ind_prod_monthly_qtd",
          "ind_prod_3episodes",
          "ind_prod_5episodes",
          "NOAA_HWI_mean",
          "NOAA_HWI_max",
          "NOAA_HWI_min",
          "caution_qtd_hr_month",
          "extreme_caution_qtd_hr_month",
          "danger_qtd_hr_month",
          "extreme_danger_qtd_hr_month",
          "noaa_qtd_monthly",
          "noaa_3episodes",
          "noaa_5episodes",
          "omm_qtd_monthly",
          "omm_3episodes",
          "omm_5episodes",
          ]

# Inverso: valor (cidade/pasta) -> chave (HW_*)
CAPITAL_TO_CODE = {v: k for k, v in CODE_TO_CAPITAL.items()}
AGG_DIR = str(CAPITAL_AGGREGATES_DIR)
Path(AGG_DIR).mkdir(parents=True, exist_ok=True)





In [ ]:
def _monthly_parquet_path(city_dir_name: str) -> str:
    """city_dir_name é o NOME DA CIDADE (pasta/arquivo)."""
    return str(ERA5_MONTHLY_DIR / city_dir_name / f"{city_dir_name}_heatwave_ERA5_monthly.parquet")
def load_city_metric_series(city_dir_name: str, metric: str) -> Optional[pd.Series]:
    """
    Lê o parquet mensal da cidade (pasta = nome da cidade) e retorna uma Série indexada em 'time'.
    A **coluna** será nomeada com o **código HW_*** correspondente (CHAVE do dicionário).
    """
    parquet_path = _monthly_parquet_path(city_dir_name)
    capital = unicodedata.normalize("NFC", city_dir_name)

    if not os.path.isfile(parquet_path):
        print(f"⚠️ Arquivo mensal não encontrado para {capital}: {parquet_path}")
        return None

    if capital not in CAPITAL_TO_CODE:
        print(f"⚠️ Cidade/pasta '{capital}' não está no mapeamento. Ignorando.")
        return None

    code = CAPITAL_TO_CODE[capital]  # ex: "HW_RJ"

    try:
        df = pd.read_parquet(parquet_path)
    except Exception as e:
        print(f"⚠️ Falha ao ler {parquet_path}: {e}")
        return None

    if "time" not in df.columns:
        print(f"⚠️ Coluna 'time' ausente em {parquet_path}. Ignorando.")
        return None
    if metric not in df.columns:
        print(f"⚠️ Métrica '{metric}' não encontrada em {parquet_path}. Ignorando.")
        return None

    s = df[["time", metric]].copy()
    s["time"] = pd.to_datetime(s["time"])
    s = s.drop_duplicates(subset="time").sort_values("time").set_index("time")[metric]

    # >>> Nome da coluna = CHAVE (HW_*)
    s.name = code
    return s

def build_wide_dataframe(metric: str, city_dirs: list[str]) -> pd.DataFrame:
    """
    Constrói um DataFrame wide para 'metric' com índice 'time' (MS) e colunas = códigos HW_*.
    Faz outer-join no tempo para cobrir datas distintas por cidade.
    """
    series_list = []
    for cd in sorted(city_dirs):
        s = load_city_metric_series(cd, metric)
        if s is not None:
            series_list.append(s)

    if not series_list:
        return pd.DataFrame(columns=["time"]).set_index(pd.DatetimeIndex([], name="time"))

    wide = pd.concat(series_list, axis=1)  # outer join por índice time
    wide.index.name = "time"
    wide = wide.sort_index()

    # opcional: garantir frequência MS quando o índice permitir
    # (mantém datas originais; descomente se quiser regularizar)
    # wide = wide.asfreq("MS")

    return wide


def save_metric_parquet(metric: str, wide_df: pd.DataFrame) -> str:
    """
    Salva o DataFrame wide em parquet com nome padronizado por métrica.
    Retorna o caminho salvo.
    """
    out_name = f"agg_{metric}.parquet"
    out_path = os.path.join(AGG_DIR, out_name)
    # Salva com time como coluna para interoperabilidade
    out_df = wide_df.reset_index()
    out_df.to_parquet(out_path, index=False)
    return out_path


def run_all_aggregations(city_dirs: list[str], metrics: list[str]) -> None:
    """
    Gera e salva todos os Parquets wide (um por métrica) na pasta _aggregations.
    """
    print(f"🔧 Iniciando agregações para {len(metrics)} métricas…")
    for m in metrics:
        print(f"➡️  Métrica: {m}")
        wide = build_wide_dataframe(m, city_dirs)
        if wide.empty:
            print(f"   ⚠️ Sem dados para '{m}'. Pulando.")
            continue
        out = save_metric_parquet(m, wide)
        print(f"   ✅ Salvo: {out}")

cities = [unicodedata.normalize("NFC", c) for c in cities]
unknown = [c for c in cities if c not in CAPITAL_TO_CODE]
if unknown:
    print("⚠️ Pastas sem mapeamento HW_*:", unknown)





In [ ]:
# Rode a agregação
run_all_aggregations(cities, METRICS)





In [ ]:
pd.set_option('display.max_columns', None)
pth = str(CAPITAL_AGGREGATES_DIR / "agg_ind_prod_3episodes.parquet")
df = pd.read_parquet(pth)
df



